# 08. From points to surfaces

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/08-point-clouds.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=08-point-clouds.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/08-point-clouds.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `7c4fc6fd3cb4c738780286bc0f2cda3a8fc53b13c038a1f499a767e3849d6245`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 8: From points to surfaces; abstract contour illustration](assets/08-point-clouds.svg)

# 08. From points to surfaces

A synthetic lidar laboratory

**Learning goals:** Write and read a LAS file; filter ground points; compare ground and canopy elevations in 3D.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=08-point-clouds.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Returns are samples, not pixels

Our point cloud is generated deterministically. Ground and canopy labels are assigned by construction; no classification algorithm has inferred them. Class 2 and class 5 follow conventional LAS ground and high-vegetation codes for this demonstration.

In [ ]:
rng = np.random.default_rng(42)
n = 1600
px = 500000 + rng.uniform(0, 1000, n)
py = 4400000 + rng.uniform(0, 1000, n)
ground_z = 1200 + .03*(px-500000) + 12*np.sin((py-4400000)/150)
classes = np.where(rng.random(n) < .65, 2, 5).astype("uint8")
pz = ground_z + np.where(classes == 5, rng.uniform(8, 25, n), 0)
cloud = terrio.PointCloud.from_dict({"x": px, "y": py, "z": pz, "classification": classes}, crs=CRS)
path = OUTPUT / "synthetic.las"
cloud.write(str(path))
back = terrio.read_points(str(path))
np.testing.assert_allclose(back["z"], pz, atol=.02)
print("Points:", len(back["z"]), "file bytes:", path.stat().st_size)

## Filter before gridding

A terrain model should represent the ground. A surface model may include roofs and canopy. Selecting labels only works as well as the classification. Real surveys also have outliers, variable density, and occluded ground.

In [ ]:
ground = terrio.read_points(str(path), classes=[2])
assert np.all(ground["classification"] == 2)
print("Ground points:", len(ground["z"]))
grid = ground.grid(50.0, stat="mean")
values = grid.read(1).astype(float)
if grid.nodata is not None:
    values[values == grid.nodata] = np.nan
image_grid([values], ["Mean ground elevation in 50 m cells"])
plt.show()

## Inspect the vertical structure

The scene distinguishes known synthetic classes and reports coordinates in metres. Orbit the cloud to see why a 2D map can hide the gap between ground and canopy. This figure has no external imagery or tile dependency.

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
for code, name, color in [(2, "Assigned ground", "#008b8b"), (5, "Assigned canopy", "#d79522")]:
    selected = classes == code
    fig.add_trace(go.Scatter3d(x=px[selected], y=py[selected], z=pz[selected], mode="markers", name=name,
                               marker=dict(size=2, color=color)))
fig.update_layout(title="Synthetic point-cloud scene", scene=dict(xaxis_title="Easting (m)", yaxis_title="Northing (m)",
                       zaxis_title="Elevation (m)", aspectmode="data"))
interactive(fig)

## Check density

A mean in a sparsely sampled cell is less stable than one supported by many points. Counts are a first diagnostic; they are not a full uncertainty estimate because points can be correlated.

In [ ]:
counts = ground.grid(50.0, stat="count").read(1)
image_grid([counts], ["Ground returns per 50 m cell"], ["viridis"], units="points")
plt.show()

## Your investigation

Compare 25 m and 100 m grids. Explain the tradeoff between detail and sampling support. Try gridding all returns instead of just ground.

### Discussion guide

Smaller cells retain detail but have fewer samples and more gaps. Including canopy raises some cell summaries and changes what the surface represents.

## Sources and credit

- [Source 1](https://terrio.dev/point-clouds/)
- [Source 2](https://terrio.dev/python/#point-clouds)
- [Source 3](https://github.com/ASPRSorg/LAS)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.